# DeepWeeds Lab Day 2

Select **GPU T4 x2** and enable **Internet** in Kaggle Notebook settings, then use **Save & Run All**. Outputs are written to `/kaggle/working/lab_output`. The smoke stage opens train and validation only. The full stage chooses the recipe and inference on validation before evaluating test. For a retry, add the previous run's `deepweeds_evidence.zip` as a Kaggle Input; the notebook will reuse all completed runs whose configs match.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'Enable GPU in Kaggle settings'
assert torch.cuda.device_count() >= 2, 'Select T4 x2 in Kaggle settings'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
%pip -q install timm==1.0.30 fvcore==0.1.5.post20221221 openpyxl

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/eval.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python3\n"""eval.py - công cụ đánh giá chính thức của Lab Day 2 (DeepWeeds).\n\nFile này đã hoàn chỉnh: sinh viên KHÔNG sửa công thức chỉ số. Mọi con số trong\nresults.xlsx và báo cáo phải khớp với kết quả của file này (RUBRIC.md, mục I).\nChỉ phụ thuộc numpy và pandas.\n\nĐịnh nghĩa chỉ số (README.md, mục 2.2)\n--------------------------------------\n- top-1 accuracy : số ảnh đoán đúng / tổng số ảnh, không trọng số theo lớp.\n- macro-F1       : trung bình cộng F1 của 9 lớp (lớp không có dự đoán đúng có F1 = 0).\n- balanced acc.  : trung bình recall của các lớp có ảnh trong tập.\n- precision / recall / F1 theo lớp, ma trận nhầm lẫn (hàng = nhãn thật).\n- ECE            : 15 bin đều theo độ tin cậy (= max softmax); bin m là (b_{m-1}, b_m].\n                   ECE = sum_m (n_m / n) * |acc_m - conf_m|\n- mean +- std    : qua các seed, std mẫu (ddof = 1).\n\nĐịnh dạng file dự đoán (mỗi seed một file)\n------------------------------------------\n    Filename, y_true, y_pred, p0, p1, ..., p8\np0..p8 là xác suất softmax, thứ tự theo cột `Label` của labels.csv\n(0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives). Tên file chứa `seed<k>`,\nví dụ predictions/F01_seed0_test.csv. Hàm `save_predictions(...)` trong file này ghi\nđúng định dạng đó (`from eval import save_predictions`).\n\nCách dùng\n---------\n    # 1) Tính chỉ số cho một nhóm file (một cấu hình, nhiều seed)\n    python eval.py score --pred "predictions/F01_seed*_test.csv" \\\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv --tag F01\n\n    # 2) Tự chấm phần I của RUBRIC (chung kết so với mốc)\n    python eval.py grade \\\n        --final    "predictions/F01_seed*_test.csv" \\\n        --baseline "predictions/T00_seed*_test.csv" \\\n        --uncal    "predictions/F01_uncal_seed*_test.csv" \\\n        --final-val "predictions/F01_seed*_val.csv" \\\n        --latency-p95-ms 41.5 --latency-method proper \\\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv\n\n`--uncal` (cùng cấu hình nhưng chưa temperature scaling) và `--final-val` (dự đoán\ntrên val) là tùy chọn; thiếu thì các ý I4(a) hoặc I4(b) ghi là "chưa chấm được".\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport glob\nimport json\nimport math\nimport re\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------- #\n# Hằng số\n# --------------------------------------------------------------------------- #\nNUM_CLASSES = 9\nECE_BINS = 15\nPROB_SUM_TOL = 1e-3  # xác suất mỗi dòng phải cộng bằng 1 (sai số cho phép)\n\n# Thứ tự lớp theo cột `Label` của labels.csv (lấy từ deepweeds.py của tác giả).\nCLASS_NAMES = [\n    "Chinee Apple", "Lantana", "Parkinsonia", "Parthenium", "Prickly Acacia",\n    "Rubber Vine", "Siam Weed", "Snake Weed", "Negatives",\n]\n\n# Hai lớp khó nhất và mốc recall (%) từ bài báo gốc (README.md, mục 2.3).\nHARD_CLASSES = {"Chinee Apple": 88.5, "Snake Weed": 88.8}\n\n# ---- Ngưỡng chấm RUBRIC mục I (TẠM THỜI: giảng viên có thể chỉnh tại đây) ---- #\nACC_TIERS = [(95.7, 7), (95.1, 6), (94.0, 5), (92.0, 3), (90.0, 1)]  # I1: (acc %, điểm)\nI2_MIN_DELTA = 0.01          # I2: Delta macro-F1 tối thiểu để đạt 5 điểm\nHARD_FLOORS = [(85.0, 3), (80.0, 2)]  # I3: sàn recall (%) cho cả hai lớp khó\nI3_MIN_POINTS = 1            # I3: có báo cáo đầy đủ nhưng dưới 80%\nI4_MAX_VAL_TEST_GAP = 0.02   # I4(b): |macro-F1 val - macro-F1 test| tối đa\nI5_P95_BUDGET_MS = 100.0     # I5: ngân sách p95 ở batch 1\nMIN_SEEDS = 3\n\n\n# --------------------------------------------------------------------------- #\n# Đọc và kiểm tra file dự đoán\n# --------------------------------------------------------------------------- #\n@dataclass\nclass Pred:\n    path: str\n    seed: int | None\n    filenames: np.ndarray\n    y_true: np.ndarray\n    y_pred: np.ndarray\n    probs: np.ndarray\n\n\ndef parse_seed(path: str) -> int | None:\n    m = re.search(r"seed[_-]?(\\d+)", Path(path).name)\n    return int(m.group(1)) if m else None\n\n\ndef read_pred(path: str, num_classes: int = NUM_CLASSES) -> Pred:\n    """Đọc một file dự đoán và kiểm tra định dạng. Ném ValueError nếu sai."""\n    df = pd.read_csv(path)\n    prob_cols = [f"p{i}" for i in range(num_classes)]\n    missing = [c for c in ["Filename", "y_true", "y_pred", *prob_cols] if c not in df.columns]\n    if missing:\n        raise ValueError(f"{path}: thiếu cột {missing}")\n    extra_p = [c for c in df.columns if re.fullmatch(r"p\\d+", c) and c not in prob_cols]\n    if extra_p:\n        raise ValueError(f"{path}: có cột xác suất ngoài p0..p{num_classes - 1}: {extra_p}")\n    if len(df) == 0:\n        raise ValueError(f"{path}: file rỗng")\n    if df["Filename"].duplicated().any():\n        raise ValueError(f"{path}: có Filename bị trùng")\n\n    probs = df[prob_cols].to_numpy(dtype=np.float64)\n    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1 + 1e-6).any():\n        raise ValueError(f"{path}: xác suất chứa NaN, âm hoặc lớn hơn 1; hãy lưu softmax, không lưu logit")\n    bad_sum = np.abs(probs.sum(1) - 1.0) > PROB_SUM_TOL\n    if bad_sum.any():\n        raise ValueError(f"{path}: {int(bad_sum.sum())} dòng có tổng xác suất khác 1 "\n                         f"(sai số > {PROB_SUM_TOL}); hãy lưu softmax, không lưu logit")\n\n    ints = {}\n    for col in ("y_true", "y_pred"):\n        v = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)\n        if not np.isfinite(v).all() or (v != np.round(v)).any():\n            raise ValueError(f"{path}: cột {col} phải là số nguyên")\n        v = v.astype(np.int64)\n        if v.min() < 0 or v.max() >= num_classes:\n            raise ValueError(f"{path}: cột {col} ngoài khoảng 0..{num_classes - 1}")\n        ints[col] = v\n\n    argmax = probs.argmax(1)\n    mismatch = int((argmax != ints["y_pred"]).sum())\n    if mismatch:\n        raise ValueError(f"{path}: {mismatch} dòng có y_pred khác argmax của p0..p{num_classes - 1}")\n\n    return Pred(path, parse_seed(path), df["Filename"].to_numpy(), ints["y_true"], ints["y_pred"], probs)\n\n\ndef save_predictions(path: str | Path, filenames, y_true, probs, num_classes: int = NUM_CLASSES) -> Path:\n    """Ghi một file dự đoán đúng định dạng của eval.py: Filename, y_true, y_pred, p0..p{K-1}.\n\n    Dùng hàm này (hoặc ghi y hệt định dạng) để tạo predictions/<exp_id>_seed<k>_<split>.csv.\n    `probs` là XÁC SUẤT softmax (mỗi dòng cộng bằng 1), không phải logit.\n    y_pred được tính bằng argmax của probs. Kết quả luôn qua được `read_pred`.\n    """\n    probs = np.asarray(probs, dtype=np.float64)\n    y_true = np.asarray(y_true, dtype=np.int64)\n    filenames = list(filenames)\n    if probs.ndim != 2 or probs.shape[1] != num_classes:\n        raise ValueError(f"probs phải có dạng (N, {num_classes}), nhận {probs.shape}")\n    if not (len(filenames) == len(y_true) == len(probs)):\n        raise ValueError("filenames, y_true, probs phải cùng độ dài")\n    if np.abs(probs.sum(1) - 1.0).max() > PROB_SUM_TOL:\n        raise ValueError("probs chưa chuẩn hoá (tổng mỗi dòng khác 1); hãy áp dụng softmax trước khi lưu")\n    df = pd.DataFrame({"Filename": filenames, "y_true": y_true, "y_pred": probs.argmax(1)})\n    for i in range(num_classes):\n        df[f"p{i}"] = probs[:, i]\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False, float_format="%.8g")\n    return path\n\n\ndef check_against_csv(pred: Pred, csv_path: str, what: str = "test") -> None:\n    """Đối chiếu tên file và nhãn thật với file CSV chia sẵn (Filename, Label)."""\n    ref = pd.read_csv(csv_path)\n    if not {"Filename", "Label"} <= set(ref.columns):\n        raise ValueError(f"{csv_path}: cần cột Filename và Label")\n    ref_map = dict(zip(ref["Filename"], ref["Label"].astype(int)))\n    names = set(pred.filenames)\n    if names != set(ref_map):\n        only_pred = len(names - set(ref_map))\n        only_ref = len(set(ref_map) - names)\n        raise ValueError(f"{pred.path}: không khớp {what} CSV ({csv_path}): "\n                         f"thừa {only_pred} ảnh, thiếu {only_ref} ảnh")\n    wrong = sum(int(ref_map[f]) != int(t) for f, t in zip(pred.filenames, pred.y_true))\n    if wrong:\n        raise ValueError(f"{pred.path}: {wrong} dòng có y_true khác Label trong {csv_path}")\n\n\ndef load_names(labels_csv: str | None) -> list[str]:\n    """Lấy tên lớp từ labels.csv (cột Label, Species) nếu có; nếu không dùng mặc định."""\n    if not labels_csv:\n        return list(CLASS_NAMES)\n    df = pd.read_csv(labels_csv)\n    if not {"Label", "Species"} <= set(df.columns):\n        raise ValueError(f"{labels_csv}: cần cột Label và Species")\n    m = df.drop_duplicates("Label").sort_values("Label")\n    if list(m["Label"]) != list(range(NUM_CLASSES)):\n        raise ValueError(f"{labels_csv}: Label phải là 0..{NUM_CLASSES - 1}")\n    return [str(s) for s in m["Species"]]\n\n\ndef _norm(name: str) -> str:\n    return re.sub(r"[^a-z]", "", name.lower())\n\n\ndef hard_class_indices(names: list[str]) -> dict[str, int]:\n    out = {}\n    for hard in HARD_CLASSES:\n        for i, n in enumerate(names):\n            if _norm(n) == _norm(hard):\n                out[hard] = i\n    missing = set(HARD_CLASSES) - set(out)\n    if missing:\n        raise ValueError(f"không tìm thấy lớp {sorted(missing)} trong danh sách tên lớp {names}")\n    return out\n\n\ndef expand(pattern: str | list[str]) -> list[str]:\n    patterns = [pattern] if isinstance(pattern, str) else list(pattern)\n    files: list[str] = []\n    for p in patterns:\n        files.extend(glob.glob(p))\n    files = sorted(set(files))\n    if not files:\n        raise FileNotFoundError(f"không có file nào khớp: {patterns}")\n    return files\n\n\n# --------------------------------------------------------------------------- #\n# Chỉ số\n# --------------------------------------------------------------------------- #\ndef confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, k: int = NUM_CLASSES) -> np.ndarray:\n    cm = np.zeros((k, k), dtype=np.int64)\n    np.add.at(cm, (y_true, y_pred), 1)\n    return cm\n\n\ndef per_class(cm: np.ndarray) -> dict[str, np.ndarray]:\n    tp = np.diag(cm).astype(np.float64)\n    support = cm.sum(1).astype(np.float64)\n    predicted = cm.sum(0).astype(np.float64)\n    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)\n    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)\n    denom = precision + recall\n    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)\n    return {"precision": precision, "recall": recall, "f1": f1, "support": support}\n\n\ndef ece_score(probs: np.ndarray, y_true: np.ndarray, bins: int = ECE_BINS) -> float:\n    """ECE với `bins` bin đều; độ tin cậy = max softmax; bin m là (b_{m-1}, b_m]."""\n    conf = probs.max(1)\n    correct = (probs.argmax(1) == y_true).astype(np.float64)\n    idx = np.clip(np.ceil(conf * bins).astype(np.int64) - 1, 0, bins - 1)\n    n = len(conf)\n    ece = 0.0\n    for m in range(bins):\n        mask = idx == m\n        if mask.any():\n            ece += mask.sum() / n * abs(correct[mask].mean() - conf[mask].mean())\n    return float(ece)\n\n\ndef compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: np.ndarray,\n                    k: int = NUM_CLASSES, bins: int = ECE_BINS) -> dict:\n    cm = confusion_matrix(y_true, y_pred, k)\n    pc = per_class(cm)\n    has_support = pc["support"] > 0\n    nll = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, None)).mean())\n    return {\n        "n": int(len(y_true)),\n        "top1": float((y_true == y_pred).mean()),\n        "macro_f1": float(pc["f1"].mean()),\n        "balanced_acc": float(pc["recall"][has_support].mean()),\n        "ece": ece_score(probs, y_true, bins),\n        "nll": nll,\n        "precision": pc["precision"], "recall": pc["recall"], "f1": pc["f1"],\n        "support": pc["support"], "confusion": cm,\n    }\n\n\nSCALARS = ("top1", "macro_f1", "balanced_acc", "ece", "nll")\nVECTORS = ("precision", "recall", "f1")\n\n\ndef mean_std(values) -> tuple[float, float]:\n    """mean và std mẫu (ddof=1) theo trục seed; std là NaN nếu chỉ có 1 seed."""\n    a = np.asarray(values, dtype=np.float64)\n    mean = a.mean(axis=0)\n    std = a.std(axis=0, ddof=1) if len(a) > 1 else np.full(a.shape[1:], np.nan)\n    if a.ndim == 1:\n        return float(mean), float(std)\n    return mean, std\n\n\n@dataclass\nclass Group:\n    preds: list[Pred]\n    metrics: list[dict]\n    summary: dict  # {"top1": (mean, std), ..., "recall": (vec_mean, vec_std), ...}\n\n    @property\n    def seeds(self) -> list[int | None]:\n        return [p.seed for p in self.preds]\n\n\ndef load_group(pattern, ref_csv: str | None, k: int = NUM_CLASSES, ref_what: str = "test") -> Group:\n    files = expand(pattern)\n    preds = [read_pred(f, k) for f in files]\n    if ref_csv:\n        for p in preds:\n            check_against_csv(p, ref_csv, ref_what)\n    base_names = set(preds[0].filenames)\n    for p in preds[1:]:\n        if set(p.filenames) != base_names:\n            raise ValueError(f"{p.path}: tập Filename khác {preds[0].path}")\n    seeds = [p.seed for p in preds if p.seed is not None]\n    if len(seeds) != len(set(seeds)):\n        raise ValueError(f"seed bị trùng giữa các file: {files}")\n    metrics = [compute_metrics(p.y_true, p.y_pred, p.probs, k) for p in preds]\n    summary = {key: mean_std([m[key] for m in metrics]) for key in (*SCALARS, *VECTORS)}\n    return Group(preds, metrics, summary)\n\n\n# --------------------------------------------------------------------------- #\n# Định dạng đầu ra\n# --------------------------------------------------------------------------- #\ndef fmt(mean: float, std: float, digits: int = 4) -> str:\n    if std is None or (isinstance(std, float) and math.isnan(std)):\n        return f"{mean:.{digits}f} (1 seed)"\n    return f"{mean:.{digits}f} ± {std:.{digits}f}"\n\n\ndef report_group(name: str, g: Group, names: list[str]) -> str:\n    lines = [f"### {name} ({len(g.preds)} seed: {g.seeds}; {g.metrics[0][\'n\']} ảnh)", ""]\n    if len(g.preds) < MIN_SEEDS:\n        lines += [f"> CẢNH BÁO: chỉ có {len(g.preds)} seed, yêu cầu tối thiểu {MIN_SEEDS}.", ""]\n    lines += ["| Chỉ số | mean ± std |", "|---|---|"]\n    labels = {"top1": "top-1 accuracy", "macro_f1": "macro-F1", "balanced_acc": "balanced accuracy",\n              "ece": "ECE (15 bin)", "nll": "NLL"}\n    for key in SCALARS:\n        lines.append(f"| {labels[key]} | {fmt(*g.summary[key])} |")\n    lines += ["", "| Lớp | Precision | Recall | F1 | Số ảnh |", "|---|---|---|---|---|"]\n    pm, ps = g.summary["precision"]\n    rm, rs = g.summary["recall"]\n    fm, fs = g.summary["f1"]\n    for i, n in enumerate(names):\n        lines.append(f"| {n} | {fmt(pm[i], ps[i], 3)} | {fmt(rm[i], rs[i], 3)} | "\n                     f"{fmt(fm[i], fs[i], 3)} | {int(g.metrics[0][\'support\'][i])} |")\n    return "\\n".join(lines)\n\n\ndef save_group(out_dir: Path, tag: str, g: Group, names: list[str]) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for p, m in zip(g.preds, g.metrics):\n        rows.append({"file": Path(p.path).name, "seed": p.seed,\n                     **{k: m[k] for k in ("n", *SCALARS)}})\n    pd.DataFrame(rows).to_csv(out_dir / f"{tag}_per_seed.csv", index=False)\n    pc = pd.DataFrame({"class": names})\n    for key in VECTORS:\n        mean, std = g.summary[key]\n        pc[f"{key}_mean"], pc[f"{key}_std"] = mean, std\n    pc["support"] = g.metrics[0]["support"].astype(int)\n    pc.to_csv(out_dir / f"{tag}_per_class.csv", index=False)\n    cm = sum(m["confusion"] for m in g.metrics)\n    pd.DataFrame(cm, index=[f"true_{n}" for n in names],\n                 columns=[f"pred_{n}" for n in names]).to_csv(out_dir / f"{tag}_confusion_sum.csv")\n    summary = {"seeds": g.seeds,\n               **{k: {"mean": g.summary[k][0], "std": g.summary[k][1]} for k in SCALARS},\n               **{k: {"mean": g.summary[k][0].tolist(), "std": g.summary[k][1].tolist()} for k in VECTORS},\n               "classes": names}\n    (out_dir / f"{tag}_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))\n\n\n# --------------------------------------------------------------------------- #\n# Lệnh `score`\n# --------------------------------------------------------------------------- #\ndef cmd_score(args) -> int:\n    names = load_names(args.labels)\n    g = load_group(args.pred, args.test_csv, ref_what="test")\n    if not args.test_csv:\n        print("LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\n")\n    print(report_group(args.tag, g, names))\n    if args.out:\n        save_group(Path(args.out), args.tag, g, names)\n        print(f"\\nĐã lưu kết quả vào {args.out}/{args.tag}_*")\n    return 0\n\n\n# --------------------------------------------------------------------------- #\n# Lệnh `grade` (RUBRIC mục I)\n# --------------------------------------------------------------------------- #\ndef points_i1(acc_pct: float) -> int:\n    for thr, pts in ACC_TIERS:\n        if acc_pct >= thr:\n            return pts\n    return 0\n\n\ndef points_i2(delta: float, s: float) -> int:\n    if not math.isfinite(delta) or delta <= 0:\n        return 0\n    s = 0.0 if not math.isfinite(s) else s\n    if delta > s and delta >= I2_MIN_DELTA:\n        return 5\n    return 4 if delta > s else 2\n\n\ndef points_i3(recalls_pct: dict[str, float]) -> int:\n    if all(recalls_pct[c] >= ref for c, ref in HARD_CLASSES.items()):\n        return 4\n    low = min(recalls_pct.values())\n    for floor, pts in HARD_FLOORS:\n        if low >= floor:\n            return pts\n    return I3_MIN_POINTS\n\n\ndef cmd_grade(args) -> int:\n    names = load_names(args.labels)\n    hard_idx = hard_class_indices(names)\n    final = load_group(args.final, args.test_csv, ref_what="test")\n    base = load_group(args.baseline, args.test_csv, ref_what="test")\n    if not args.test_csv:\n        print("LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\n")\n    warnings: list[str] = []\n    for label, g in (("chung kết", final), ("mốc", base)):\n        if len(g.preds) < MIN_SEEDS:\n            warnings.append(f"nhóm {label} chỉ có {len(g.preds)} seed (< {MIN_SEEDS})")\n    if set(final.preds[0].filenames) != set(base.preds[0].filenames):\n        warnings.append("tập Filename của chung kết và mốc khác nhau")\n\n    items: list[tuple[str, str, int | None, int, str]] = []  # (mã, tiêu chí, điểm, tối đa, ghi chú)\n\n    # I1\n    acc = final.summary["top1"][0] * 100\n    items.append(("I1", "Top-1 accuracy test", points_i1(acc), 7, f"{acc:.2f}% (mean {len(final.preds)} seed)"))\n\n    # I2\n    mf_f, sd_f = final.summary["macro_f1"]\n    mf_b, sd_b = base.summary["macro_f1"]\n    delta = mf_f - mf_b\n    s = max([v for v in (sd_f, sd_b) if math.isfinite(v)], default=math.nan)\n    items.append(("I2", "Macro-F1 cải thiện so với mốc", points_i2(delta, s), 5,\n                  f"final {mf_f:.4f}, mốc {mf_b:.4f}, Δ={delta:+.4f}, s={s:.4f}"))\n\n    # I3\n    rec_mean = final.summary["recall"][0] * 100\n    recalls = {c: float(rec_mean[i]) for c, i in hard_idx.items()}\n    note = ", ".join(f"{c} {recalls[c]:.1f}% (mốc {HARD_CLASSES[c]}%)" for c in HARD_CLASSES)\n    items.append(("I3", "Recall hai lớp khó", points_i3(recalls), 4, note))\n\n    # I4(a)\n    if args.uncal:\n        uncal = load_group(args.uncal, args.test_csv, ref_what="test")\n        e_after, e_before = final.summary["ece"][0], uncal.summary["ece"][0]\n        items.append(("I4a", "ECE sau TS < ECE trước", int(e_after < e_before), 1,\n                      f"trước {e_before:.4f}, sau {e_after:.4f}"))\n    else:\n        items.append(("I4a", "ECE sau TS < ECE trước", None, 1, "chưa chấm được (thiếu --uncal)"))\n\n    # I4(b)\n    if args.final_val:\n        fv = load_group(args.final_val, args.val_csv, ref_what="val")\n        gap = abs(fv.summary["macro_f1"][0] - mf_f)\n        items.append(("I4b", "Chênh macro-F1 val/test <= %.2f" % I4_MAX_VAL_TEST_GAP,\n                      int(gap <= I4_MAX_VAL_TEST_GAP), 1,\n                      f"val {fv.summary[\'macro_f1\'][0]:.4f}, test {mf_f:.4f}, chênh {gap:.4f}"))\n    else:\n        items.append(("I4b", "Chênh macro-F1 val/test <= %.2f" % I4_MAX_VAL_TEST_GAP, None, 1,\n                      "chưa chấm được (thiếu --final-val)"))\n\n    # I5\n    if args.latency_p95_ms is None:\n        items.append(("I5", "Cấu hình thời gian thực", None, 2, "chưa chấm được (thiếu --latency-p95-ms)"))\n    else:\n        ok = args.latency_p95_ms <= I5_P95_BUDGET_MS\n        pts = 0 if not ok else (2 if args.latency_method == "proper" else 1)\n        items.append(("I5", "Cấu hình thời gian thực", pts, 2,\n                      f"p95 = {args.latency_p95_ms:.1f} ms (ngân sách {I5_P95_BUDGET_MS:.0f} ms), "\n                      f"đo {\'đúng cách\' if args.latency_method == \'proper\' else \'chưa đủ warmup/synchronize\'}"))\n\n    scored = [i for i in items if i[2] is not None]\n    got = sum(i[2] for i in scored)\n    max_scored = sum(i[3] for i in scored)\n    lines = ["## Tự chấm RUBRIC mục I (đề xuất; giảng viên xác nhận)", "",\n             "| Mã | Tiêu chí | Điểm | Tối đa | Chi tiết |", "|---|---|---|---|---|"]\n    for code, crit, pts, mx, note in items:\n        lines.append(f"| {code} | {crit} | {\'-\' if pts is None else pts} | {mx} | {note} |")\n    lines += ["", f"**Tổng các ý đã chấm: {got} / {max_scored}** (phần I tối đa 20)."]\n    pending = [i[0] for i in items if i[2] is None]\n    if pending:\n        lines.append(f"Chưa chấm được: {\', \'.join(pending)}.")\n    if warnings:\n        lines += ["", "Cảnh báo:"] + [f"- {w}" for w in warnings]\n    lines += ["", "Ngưỡng điểm là TẠM THỜI (xem khối hằng số đầu file eval.py và RUBRIC.md mục I)."]\n    print("\\n".join(lines))\n\n    if args.out:\n        out = Path(args.out)\n        out.mkdir(parents=True, exist_ok=True)\n        (out / "grade_I.json").write_text(json.dumps({\n            "items": [{"code": c, "criterion": cr, "points": p, "max": m, "note": n} for c, cr, p, m, n in items],\n            "total": got, "max_scored": max_scored, "warnings": warnings}, indent=2, ensure_ascii=False))\n    return 0\n\n\n# --------------------------------------------------------------------------- #\n# CLI\n# --------------------------------------------------------------------------- #\ndef build_parser() -> argparse.ArgumentParser:\n    ap = argparse.ArgumentParser(description="Đánh giá Lab Day 2 (DeepWeeds)")\n    sub = ap.add_subparsers(dest="cmd", required=True)\n\n    sc = sub.add_parser("score", help="tính chỉ số cho một nhóm file dự đoán (nhiều seed)")\n    sc.add_argument("--pred", nargs="+", required=True, help="glob của file dự đoán")\n    sc.add_argument("--test-csv", help="labels/test_subset0.csv để đối chiếu tên file và nhãn")\n    sc.add_argument("--labels", help="labels/labels.csv để lấy tên lớp")\n    sc.add_argument("--tag", default="score")\n    sc.add_argument("--out", help="thư mục lưu JSON/CSV")\n    sc.set_defaults(func=cmd_score)\n\n    gr = sub.add_parser("grade", help="tự chấm phần I của RUBRIC")\n    gr.add_argument("--final", nargs="+", required=True, help="glob dự đoán test của cấu hình chung kết")\n    gr.add_argument("--baseline", nargs="+", required=True, help="glob dự đoán test của mốc (T00 + I00)")\n    gr.add_argument("--uncal", nargs="+", help="glob dự đoán test của chung kết khi chưa temperature scaling")\n    gr.add_argument("--final-val", nargs="+", help="glob dự đoán val của chung kết")\n    gr.add_argument("--latency-p95-ms", type=float, help="p95 batch 1 của cấu hình thời gian thực (ms)")\n    gr.add_argument("--latency-method", choices=["proper", "partial"], default="proper",\n                    help="proper = có warmup và synchronize; partial = chưa đủ")\n    gr.add_argument("--test-csv", help="labels/test_subset0.csv")\n    gr.add_argument("--val-csv", help="labels/val_subset0.csv (đối chiếu --final-val)")\n    gr.add_argument("--labels", help="labels/labels.csv")\n    gr.add_argument("--out", help="thư mục lưu grade_I.json")\n    gr.set_defaults(func=cmd_grade)\n    return ap\n\n\ndef main(argv=None) -> int:\n    args = build_parser().parse_args(argv)\n    try:\n        return args.func(args)\n    except (ValueError, FileNotFoundError) as e:\n        print(f"LỖI: {e}", file=sys.stderr)\n        return 2\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/dataset.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""DeepWeeds fold-0 data loading and transforms."""\nfrom pathlib import Path\nimport random\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\nfrom torchvision import transforms as T\n\nNUM_CLASSES = 9\nCLASS_NAMES = ["Chinee Apple", "Lantana", "Parkinsonia", "Parthenium", "Prickly Acacia", "Rubber Vine", "Siam Weed", "Snake Weed", "Negatives"]\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\ndef load_split(labels_dir, fold=0):\n    if fold != 0:\n        raise ValueError("Lab requires author\'s fold 0")\n    return tuple(pd.read_csv(Path(labels_dir) / f"{part}_subset{fold}.csv") for part in ("train", "val", "test"))\n\ndef check_split(train_df, val_df, test_df, images_dir):\n    groups = {"train": train_df, "val": val_df, "test": test_df}\n    names = {}\n    for part, frame in groups.items():\n        if not {"Filename", "Label"}.issubset(frame.columns):\n            raise ValueError(f"{part}: missing columns")\n        if frame.Filename.isna().any() or frame.Filename.duplicated().any():\n            raise ValueError(f"{part}: duplicate/null filename")\n        if not np.isin(frame.Label.to_numpy(), np.arange(NUM_CLASSES)).all():\n            raise ValueError(f"{part}: invalid labels")\n        names[part] = set(frame.Filename)\n    overlap = {f"{a}_{b}": len(names[a] & names[b]) for a, b in (("train", "val"), ("train", "test"), ("val", "test"))}\n    union = set.union(*names.values())\n    if any(overlap.values()) or len(union) != 17509:\n        raise ValueError("Fold 0 overlapping or incomplete")\n    missing = [n for n in union if not (Path(images_dir) / n).is_file()]\n    if missing:\n        raise FileNotFoundError(f"{len(missing)} images missing; first={missing[0]}")\n    return {"n": {k: len(v) for k, v in groups.items()}, "overlap": overlap,\n            "per_class": {k: {str(c): int((v.Label == c).sum()) for c in range(NUM_CLASSES)} for k, v in groups.items()}}\n\ndef build_transforms(train, img_size=224, aug="basic"):\n    if not train:\n        return T.Compose([T.Resize(256), T.CenterCrop(img_size), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])\n    ops = [T.RandomResizedCrop(img_size, scale=(0.7, 1.0)), T.RandomHorizontalFlip()]\n    if aug == "color":\n        ops.append(T.ColorJitter(0.2, 0.2, 0.2, 0.05))\n    elif aug == "trivial":\n        ops.append(T.TrivialAugmentWide())\n    elif aug == "randaug":\n        ops.append(T.RandAugment(2, 7))\n    elif aug != "basic":\n        raise ValueError(f"Unknown aug: {aug}")\n    return T.Compose([*ops, T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])\n\nclass DeepWeedsDataset(Dataset):\n    def __init__(self, df, images_dir, transform=None):\n        self.df = df.reset_index(drop=True)\n        self.root = Path(images_dir)\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, i):\n        row = self.df.iloc[i]\n        with Image.open(self.root / row.Filename) as im:\n            image = im.convert("RGB")\n        return (self.transform(image) if self.transform else image, int(row.Label), str(row.Filename))\n\ndef _seed_worker(_):\n    seed = torch.initial_seed() % (2 ** 32)\n    random.seed(seed)\n    np.random.seed(seed)\n\ndef make_loader(df, images_dir, transform, batch_size, train, sampler=None, num_workers=2):\n    if sampler not in (None, "balanced") or (sampler and not train):\n        raise ValueError(f"Invalid sampler: {sampler}")\n    weighted = None\n    if sampler == "balanced":\n        counts = df.Label.value_counts()\n        weighted = WeightedRandomSampler([1 / counts[int(y)] for y in df.Label], len(df), replacement=True)\n    return DataLoader(DeepWeedsDataset(df, images_dir, transform), batch_size=batch_size,\n                      shuffle=bool(train and weighted is None), sampler=weighted,\n                      drop_last=bool(train and len(df) > batch_size), num_workers=num_workers,\n                      pin_memory=torch.cuda.is_available(), worker_init_fn=_seed_worker)\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/model.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Backbones, classifier heads, and parameter groups."""\nimport torch\nimport timm\n\nSUGGESTED_BACKBONES = {"resnet50": "resnet50", "resnext50": "resnext50_32x4d", "convnext_tiny": "convnext_tiny", "deit_small": "deit_small_patch16_224", "swin_tiny": "swin_tiny_patch4_window7_224", "efficientnet_b0": "efficientnet_b0", "mobilenetv3": "mobilenetv3_large_100"}\n\ndef _head_names(model):\n    ids = {id(p) for p in model.get_classifier().parameters()}\n    return {name for name, p in model.named_parameters() if id(p) in ids}\n\ndef build_model(name, pretrained=True, num_classes=9, drop_rate=0.0, init="finetune"):\n    if init not in ("scratch", "frozen", "finetune"):\n        raise ValueError(f"Unknown init: {init}")\n    net = timm.create_model(name, pretrained=bool(pretrained and init != "scratch"), num_classes=num_classes, drop_rate=drop_rate)\n    if init == "frozen":\n        freeze_backbone(net)\n    return net\n\ndef freeze_backbone(model):\n    head = _head_names(model)\n    if not head:\n        raise ValueError("Classifier head not found")\n    for name, p in model.named_parameters():\n        p.requires_grad = name in head\n\ndef param_groups(model, lr_backbone, lr_head, weight_decay):\n    head = _head_names(model)\n    groups = {(h, d): [] for h in (False, True) for d in (False, True)}\n    for name, p in model.named_parameters():\n        if p.requires_grad:\n            groups[(name in head, p.ndim > 1)].append(p)\n    return [{"params": ps, "lr": lr_head if h else lr_backbone, "weight_decay": weight_decay if d else 0.0} for (h, d), ps in groups.items() if ps]\n\ndef count_params(model):\n    return sum(p.numel() for p in model.parameters()) / 1e6\n\ndef count_gmacs(model, img_size=224):\n    from fvcore.nn import FlopCountAnalysis\n    device = next(model.parameters()).device\n    was_training = model.training\n    model.eval()\n    try:\n        with torch.no_grad():\n            return FlopCountAnalysis(model, torch.zeros(1, 3, img_size, img_size, device=device)).total() / 1e9\n    finally:\n        model.train(was_training)\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/losses.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Loss functions and Mixup/CutMix."""\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass LabelSmoothingCE(nn.Module):\n    def __init__(self, smoothing=0.1):\n        super().__init__()\n        self.smoothing = smoothing\n\n    def forward(self, logits, target):\n        return F.cross_entropy(logits, target, label_smoothing=self.smoothing)\n\nclass FocalLoss(nn.Module):\n    def __init__(self, gamma=2.0, alpha=None):\n        super().__init__()\n        self.gamma = gamma\n        self.register_buffer("alpha", None if alpha is None else torch.as_tensor(alpha, dtype=torch.float))\n\n    def forward(self, logits, target):\n        log_pt = F.log_softmax(logits, dim=1).gather(1, target[:, None]).squeeze(1)\n        loss = -(1 - log_pt.exp()).pow(self.gamma) * log_pt\n        return (loss if self.alpha is None else loss * self.alpha[target]).mean()\n\ndef build_criterion(kind="ce", **kw):\n    if kind == "ce":\n        return nn.CrossEntropyLoss()\n    if kind == "ls":\n        return LabelSmoothingCE(kw.get("smoothing", 0.1))\n    if kind == "focal":\n        return FocalLoss(kw.get("gamma", 2.0), kw.get("alpha"))\n    if kind == "ce_weighted":\n        return nn.CrossEntropyLoss(weight=kw["weight"])\n    raise ValueError(f"Unknown loss: {kind}")\n\ndef class_weights(counts, beta=0.0):\n    counts = np.asarray(counts, dtype=float)\n    if np.any(counts <= 0) or not 0 <= beta < 1:\n        raise ValueError("Invalid counts or beta")\n    w = 1 / counts if beta == 0 else (1 - beta) / (1 - np.power(beta, counts))\n    return torch.tensor(w / w.mean(), dtype=torch.float)\n\ndef mix_batch(x, y, alpha=1.0, mode="cutmix"):\n    if mode not in ("mixup", "cutmix") or alpha <= 0:\n        raise ValueError("Invalid mix mode/alpha")\n    lam = float(np.random.beta(alpha, alpha))\n    perm = torch.randperm(x.size(0), device=x.device)\n    if mode == "mixup":\n        out = lam * x + (1 - lam) * x[perm]\n    else:\n        out = x.clone()\n        h, w = x.shape[-2:]\n        rw, rh = int(w * np.sqrt(1 - lam)), int(h * np.sqrt(1 - lam))\n        cx, cy = np.random.randint(w), np.random.randint(h)\n        x1, x2 = max(0, cx - rw // 2), min(w, cx + rw // 2)\n        y1, y2 = max(0, cy - rh // 2), min(h, cy + rh // 2)\n        out[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]\n        lam = 1 - (x2 - x1) * (y2 - y1) / (h * w)\n    return out, (y, y[perm], lam)\n\ndef mixed_loss(criterion, logits, targets):\n    a, b, lam = targets\n    return lam * criterion(logits, a) + (1 - lam) * criterion(logits, b)\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/train.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""One configurable training entry point for DeepWeeds experiments."""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport json\nimport math\nimport random\nimport sys\nimport time\nfrom dataclasses import asdict, dataclass, fields\nfrom pathlib import Path\nfrom typing import get_args, get_origin, get_type_hints\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nimport dataset\nimport losses\nimport model as models\n\n\n@dataclass\nclass Config:\n    exp_id: str = "T00"\n    seed: int = 0\n    fold: int = 0\n    backbone: str = "resnet50"\n    init: str = "finetune"\n    drop_rate: float = 0.0\n    img_size: int = 224\n    aug: str = "basic"\n    sampler: str | None = None\n    mix: str | None = None\n    mix_alpha: float = 1.0\n    loss: str = "ce"\n    label_smoothing: float = 0.0\n    focal_gamma: float = 2.0\n    class_weight_beta: float | None = None\n    epochs: int = 12\n    batch_size: int = 64\n    lr_backbone: float = 1e-4\n    lr_head: float = 1e-3\n    weight_decay: float = 0.05\n    warmup_epochs: float = 1.0\n    ema_decay: float | None = None\n    amp: bool = True\n    num_workers: int = 2\n    images_dir: str = "data/images"\n    labels_dir: str = "data/labels"\n    out_dir: str = "runs"\n    pred_dir: str = "predictions"\n    save_test_predictions: bool = False\n    max_train_batches: int | None = None\n    max_val_batches: int | None = None\n    use_two_gpus: bool = True\n\n\ndef run_dir(cfg):\n    return Path(cfg.out_dir) / cfg.exp_id / f"seed{cfg.seed}"\n\n\ndef pred_path(cfg, split):\n    return Path(cfg.pred_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{split}.csv"\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef build_optimizer(net, cfg):\n    return torch.optim.AdamW(models.param_groups(net, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay))\n\n\ndef build_scheduler(optimizer, cfg, steps_per_epoch):\n    total = max(1, cfg.epochs * steps_per_epoch)\n    warmup = int(cfg.warmup_epochs * steps_per_epoch)\n    def factor(step):\n        if step < warmup:\n            return max(1e-5, (step + 1) / max(1, warmup))\n        progress = (step - warmup) / max(1, total - warmup)\n        return max(1e-5, 0.5 * (1 + math.cos(math.pi * min(1, progress))))\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)\n\n\nclass EMA:\n    def __init__(self, net, decay):\n        self.model = copy.deepcopy(net).eval()\n        self.decay = decay\n        for p in self.model.parameters():\n            p.requires_grad_(False)\n\n    @torch.no_grad()\n    def update(self, net):\n        for avg, cur in zip(self.model.parameters(), net.parameters()):\n            avg.lerp_(cur.detach(), 1 - self.decay)\n        for avg, cur in zip(self.model.buffers(), net.buffers()):\n            avg.copy_(cur)\n\n\ndef train_one_epoch(net, loader, criterion, optimizer, scheduler, scaler, cfg, device, ema=None):\n    net.train()\n    if cfg.init == "frozen":\n        net.eval()\n        raw = net.module if isinstance(net, nn.DataParallel) else net\n        raw.get_classifier().train()\n    total, n = 0.0, 0\n    for step, (x, y, _) in enumerate(loader):\n        if cfg.max_train_batches is not None and step >= cfg.max_train_batches:\n            break\n        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n        targets = None\n        if cfg.mix:\n            x, targets = losses.mix_batch(x, y, cfg.mix_alpha, cfg.mix)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == "cuda"):\n            logits = net(x)\n            loss = (losses.mixed_loss(criterion, logits, targets) if targets else criterion(logits, y))\n        scaler.scale(loss).backward()\n        scale_before = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        if scaler.get_scale() >= scale_before:\n            scheduler.step()\n            if ema:\n                ema.update(net.module if isinstance(net, nn.DataParallel) else net)\n        total += float(loss.detach()) * len(y)\n        n += len(y)\n    return {"train_loss": total / n, "lr": optimizer.param_groups[0]["lr"]}\n\n\ndef evaluate(net, loader, criterion, device, max_batches=None):\n    net.eval()\n    names, labels, logits, total, n = [], [], [], 0.0, 0\n    with torch.inference_mode():\n        for step, (x, y, filenames) in enumerate(loader):\n            if max_batches is not None and step >= max_batches:\n                break\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            z = net(x)\n            total += float(criterion(z, y)) * len(y)\n            n += len(y)\n            names.extend(filenames)\n            labels.append(y.cpu().numpy())\n            logits.append(z.float().cpu().numpy())\n    return names, np.concatenate(labels), np.concatenate(logits), total / n\n\n\ndef plot_curves(history, path, title):\n    frame = pd.DataFrame(history)\n    fig, ax = plt.subplots(1, 2, figsize=(10, 4))\n    ax[0].plot(frame.epoch, frame.train_loss, label="train")\n    ax[0].plot(frame.epoch, frame.val_loss, label="val")\n    ax[0].set(ylabel="loss", xlabel="epoch")\n    ax[0].legend()\n    ax[1].plot(frame.epoch, frame.macro_f1_val, label="macro-F1 val")\n    ax[1].set(ylabel="macro-F1", xlabel="epoch", ylim=(0, 1))\n    ax[1].legend()\n    fig.suptitle(title)\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=160, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef _metrics(y, z):\n    from eval import compute_metrics\n    p = torch.softmax(torch.from_numpy(z), dim=1).numpy()\n    return compute_metrics(y, p.argmax(1), p), p\n\n\ndef run(cfg):\n    root = Path(__file__).resolve().parents[3]\n    if str(root) not in sys.path:\n        sys.path.insert(0, str(root))\n    from eval import save_predictions\n    if cfg.max_train_batches is not None and cfg.save_test_predictions:\n        raise ValueError("Smoke run must not open test")\n    set_seed(cfg.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    out = run_dir(cfg)\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=2))\n    train_df, val_df, test_df = dataset.load_split(cfg.labels_dir, cfg.fold)\n    split_info = dataset.check_split(train_df, val_df, test_df, cfg.images_dir)\n    (out / "split_check.json").write_text(json.dumps(split_info, indent=2))\n    train_loader = dataset.make_loader(train_df, cfg.images_dir, dataset.build_transforms(True, cfg.img_size, cfg.aug), cfg.batch_size, True, cfg.sampler, cfg.num_workers)\n    val_loader = dataset.make_loader(val_df, cfg.images_dir, dataset.build_transforms(False, cfg.img_size), cfg.batch_size, False, num_workers=cfg.num_workers)\n    net = models.build_model(cfg.backbone, init=cfg.init, drop_rate=cfg.drop_rate).to(device)\n    tag = ("scratch" if cfg.init == "scratch" else\n           str(net.pretrained_cfg.get("hf_hub_id") or net.pretrained_cfg.get("url") or "pretrained"))\n    params_m = models.count_params(net)\n    optimizer = build_optimizer(net, cfg)\n    train_steps = min(len(train_loader), cfg.max_train_batches) if cfg.max_train_batches else len(train_loader)\n    scheduler = build_scheduler(optimizer, cfg, train_steps)\n    counts = train_df.Label.value_counts().reindex(range(9), fill_value=0).to_numpy()\n    weights = losses.class_weights(counts, cfg.class_weight_beta or 0).to(device)\n    criterion = losses.build_criterion(cfg.loss, smoothing=cfg.label_smoothing, gamma=cfg.focal_gamma,\n                                       weight=weights, alpha=weights if cfg.loss == "focal" and cfg.class_weight_beta is not None else None).to(device)\n    val_criterion = nn.CrossEntropyLoss()\n    ema = EMA(net, cfg.ema_decay) if cfg.ema_decay is not None else None\n    wrapped = nn.DataParallel(net) if cfg.use_two_gpus and torch.cuda.device_count() > 1 else net\n    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and device.type == "cuda")\n    history, best = [], -1.0\n    for epoch in range(1, cfg.epochs + 1):\n        start = time.perf_counter()\n        tr = train_one_epoch(wrapped, train_loader, criterion, optimizer, scheduler, scaler, cfg, device, ema)\n        seconds = time.perf_counter() - start\n        eval_net = ema.model if ema else wrapped\n        _, y, z, val_loss = evaluate(eval_net, val_loader, val_criterion, device, cfg.max_val_batches)\n        metrics, _ = _metrics(y, z)\n        row = {"epoch": epoch, **tr, "val_loss": val_loss, "macro_f1_val": metrics["macro_f1"],\n               "top1_val": metrics["top1"], "seconds": seconds}\n        history.append(row)\n        pd.DataFrame(history).to_csv(out / "history.csv", index=False)\n        if row["macro_f1_val"] > best:\n            best, best_epoch = row["macro_f1_val"], epoch\n            torch.save({"model": copy.deepcopy((ema.model if ema else net).state_dict()),\n                        "epoch": epoch, "macro_f1_val": best}, out / "best.pt")\n        print(json.dumps({"exp_id": cfg.exp_id, "seed": cfg.seed, **row}), flush=True)\n    checkpoint = torch.load(out / "best.pt", map_location=device, weights_only=True)\n    net.load_state_dict(checkpoint["model"])\n    names, y, z, _ = evaluate(net, val_loader, val_criterion, device)\n    metrics, p = _metrics(y, z)\n    np.savez_compressed(out / "val_logits.npz", Filename=np.array(names), y_true=y, logits=z)\n    save_predictions(pred_path(cfg, "val"), names, y, p)\n    result = {"exp_id": cfg.exp_id, "seed": cfg.seed, "backbone": cfg.backbone,\n              "pretrained_tag": tag, "params_m": params_m, "best_epoch": best_epoch,\n              "macro_f1_val": metrics["macro_f1"], "top1_val": metrics["top1"],\n              "ece_val": metrics["ece"], "seconds_per_epoch": float(np.mean([r["seconds"] for r in history]))}\n    try:\n        result["gmac"] = models.count_gmacs(net, cfg.img_size)\n    except (ImportError, RuntimeError, ValueError) as exc:\n        result["gmac"] = None\n        result["gmac_error"] = str(exc)\n    if cfg.save_test_predictions:\n        test_loader = dataset.make_loader(test_df, cfg.images_dir, dataset.build_transforms(False, cfg.img_size), cfg.batch_size, False, num_workers=cfg.num_workers)\n        names, y, z, _ = evaluate(net, test_loader, val_criterion, device)\n        np.savez_compressed(out / "test_logits.npz", Filename=np.array(names), y_true=y, logits=z)\n        save_predictions(pred_path(cfg, "test"), names, y, torch.softmax(torch.from_numpy(z), dim=1).numpy())\n    plot_curves(history, Path(cfg.out_dir).parent / "curves" / f"{cfg.exp_id}_seed{cfg.seed}.png", f"{cfg.exp_id} {cfg.backbone} seed {cfg.seed}")\n    (out / "summary.json").write_text(json.dumps(result, indent=2))\n    return result\n\n\ndef parse_overrides(pairs):\n    hints = get_type_hints(Config)\n    result = {}\n    for pair in pairs:\n        if "=" not in pair:\n            raise ValueError(f"Expected KEY=VALUE: {pair}")\n        key, value = pair.split("=", 1)\n        if key not in hints:\n            raise ValueError(f"Unknown config field: {key}")\n        kind = hints[key]\n        if get_origin(kind) is not None:\n            kind = next((a for a in get_args(kind) if a is not type(None)), str)\n        if value.lower() in ("none", "null"):\n            result[key] = None\n        elif kind is bool:\n            if value.lower() not in ("true", "false", "1", "0"):\n                raise ValueError(f"Invalid boolean: {value}")\n            result[key] = value.lower() in ("true", "1")\n        else:\n            result[key] = kind(value)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--set", nargs="*", default=[])\n    args = parser.parse_args()\n    print(json.dumps(run(Config(**parse_overrides(args.set))), indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/inference.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Inference views, ensembles, calibration, and Conv-BN fusion."""\nimport copy\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\ndef predict_logits(model, loader, device, view=None):\n    model.eval()\n    names, labels, logits = [], [], []\n    with torch.inference_mode():\n        for x, y, filenames in loader:\n            x = x.to(device)\n            if view is not None:\n                x = view(x)\n            z = model(x)\n            names.extend(filenames)\n            labels.append(y.numpy())\n            logits.append(z.float().cpu().numpy())\n    return names, np.concatenate(labels), np.concatenate(logits)\n\ndef view_identity(x):\n    return x\n\ndef view_hflip(x):\n    return torch.flip(x, dims=(-1,))\n\ndef views_multicrop(x, crop):\n    h, w = x.shape[-2:]\n    if crop > h or crop > w:\n        raise ValueError("crop exceeds image")\n    return [x[:, :, y:y + crop, xx:xx + crop] for y, xx in\n            ((0, 0), (0, w - crop), (h - crop, 0), (h - crop, w - crop), ((h - crop)//2, (w - crop)//2))]\n\ndef views_multiscale(x, sizes):\n    return [F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False) for size in sizes]\n\ndef aggregate_views(logits_per_view, space="prob"):\n    views = np.asarray(logits_per_view)\n    if views.ndim != 3 or not len(views):\n        raise ValueError("Expected K x N x C logits")\n    if space == "prob":\n        z = views - views.max(axis=2, keepdims=True)\n        p = np.exp(z)\n        return (p / p.sum(axis=2, keepdims=True)).mean(axis=0)\n    if space == "logit":\n        z = views.mean(axis=0)\n        z -= z.max(axis=1, keepdims=True)\n        p = np.exp(z)\n        return p / p.sum(axis=1, keepdims=True)\n    raise ValueError(f"Unknown aggregation: {space}")\n\ndef ensemble_probs(list_of_probs):\n    arrays = [np.asarray(p) for p in list_of_probs]\n    if not arrays or any(a.shape != arrays[0].shape for a in arrays):\n        raise ValueError("Ensemble probabilities must have equal shapes")\n    return np.mean(arrays, axis=0)\n\ndef fit_temperature(val_logits, val_labels):\n    z = torch.as_tensor(val_logits, dtype=torch.float64)\n    y = torch.as_tensor(val_labels, dtype=torch.long)\n    log_t = torch.zeros((), dtype=torch.float64, requires_grad=True)\n    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100, line_search_fn="strong_wolfe")\n    def closure():\n        optimizer.zero_grad()\n        loss = F.cross_entropy(z / log_t.exp(), y)\n        loss.backward()\n        return loss\n    optimizer.step(closure)\n    return float(log_t.detach().exp())\n\ndef apply_temperature(logits, T):\n    if T <= 0:\n        raise ValueError("T must be positive")\n    z = np.asarray(logits, dtype=float) / T\n    z -= z.max(axis=1, keepdims=True)\n    p = np.exp(z)\n    return p / p.sum(axis=1, keepdims=True)\n\ndef fuse_conv_bn(model):\n    result = copy.deepcopy(model).eval()\n    def visit(module):\n        names = list(module._modules)\n        for prev, cur in zip(names, names[1:]):\n            a, b = module._modules[prev], module._modules[cur]\n            if isinstance(a, nn.Conv2d) and isinstance(b, nn.BatchNorm2d):\n                module._modules[prev] = torch.nn.utils.fusion.fuse_conv_bn_eval(a, b)\n                module._modules[cur] = nn.Identity()\n        for child in module.children():\n            visit(child)\n    visit(result)\n    return result\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/benchmark.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Synchronized inference latency measurements."""\nimport time\nimport numpy as np\nimport torch\n\ndef bench(fn, warmup=10, iters=100, sync=None):\n    if warmup < 10 or iters < 50:\n        raise ValueError("Requires >=10 warmup and >=50 measured iterations")\n    for _ in range(warmup):\n        fn()\n    samples = []\n    for _ in range(iters):\n        if sync:\n            sync()\n        start = time.perf_counter()\n        fn()\n        if sync:\n            sync()\n        samples.append((time.perf_counter() - start) * 1000)\n    p50, p95, p99 = np.percentile(samples, [50, 95, 99])\n    return {"p50": float(p50), "p95": float(p95), "p99": float(p99),\n            "mean": float(np.mean(samples)), "n": iters}\n\ndef latency_report(model, batch_size, img_size, dtype="fp32", device="cuda", warmup=10, iters=100):\n    dev = torch.device(device)\n    if dev.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA unavailable")\n    if dtype not in ("fp32", "amp", "fp16"):\n        raise ValueError(dtype)\n    net = model.to(dev).eval()\n    x = torch.randn(batch_size, 3, img_size, img_size, device=dev)\n    if dtype == "fp16":\n        net = net.half()\n        x = x.half()\n    def forward():\n        with torch.inference_mode(), torch.autocast(device_type=dev.type, enabled=dtype == "amp"):\n            net(x)\n    sync = (lambda: torch.cuda.synchronize(dev)) if dev.type == "cuda" else None\n    result = bench(forward, warmup, iters, sync)\n    return {**result, "gpu": torch.cuda.get_device_name(dev) if dev.type == "cuda" else "CPU",\n            "dtype": dtype, "batch": batch_size, "img_size": img_size,\n            "images_per_s": batch_size / (result["p50"] / 1000),\n            "torch": torch.__version__, "preprocessing_included": False}\n\ndef tta_latency(model, k_views, **kw):\n    if k_views < 1:\n        raise ValueError("k_views must be >=1")\n    dev = torch.device(kw.get("device", "cuda"))\n    net = model.to(dev).eval()\n    batch, size = kw.get("batch_size", 1), kw.get("img_size", 224)\n    x = torch.randn(batch, 3, size, size, device=dev)\n    def forward():\n        with torch.inference_mode():\n            for view in range(k_views):\n                net(x if view == 0 else torch.flip(x, dims=(-1,)))\n    sync = (lambda: torch.cuda.synchronize(dev)) if dev.type == "cuda" else None\n    return {**bench(forward, kw.get("warmup", 10), kw.get("iters", 100), sync), "k_views": k_views}\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/run_lab.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Reproducible DeepWeeds experiment runner for Kaggle T4 x2."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport timm\nimport torchvision\nfrom PIL import Image\n\nimport benchmark\nimport dataset\nimport inference\nimport model as models\nimport train\n\n\ndef _root():\n    return Path(__file__).resolve().parents[3]\n\n\ndef _eval():\n    if str(_root()) not in sys.path:\n        sys.path.insert(0, str(_root()))\n    import eval as ev\n    return ev\n\n\ndef _json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, default=float))\n\n\ndef _load_result(cfg):\n    directory = train.run_dir(cfg)\n    path = directory / "summary.json"\n    if path.exists():\n        saved = json.loads((directory / "config.json").read_text())\n        if saved != asdict(cfg):\n            raise ValueError(f"Existing run has a different config: {directory}")\n        return json.loads(path.read_text())\n    overrides = [f"{key}={str(value).lower() if isinstance(value, bool) or value is None else value}"\n                 for key, value in asdict(cfg).items()]\n    subprocess.run([sys.executable, str(Path(train.__file__).resolve()), "--set", *overrides], check=True)\n    return json.loads(path.read_text())\n\n\ndef restore_completed_runs(archive_path, output):\n    """Restore only runs with config, checkpoint and summary from an evidence zip."""\n    output = Path(output)\n    count = 0\n    with zipfile.ZipFile(archive_path) as saved:\n        names = set(saved.namelist())\n        complete = set()\n        for member in names:\n            if member.startswith("runs/") and member.endswith("/summary.json"):\n                directory = member.rsplit("/", 1)[0]\n                if {f"{directory}/config.json", f"{directory}/best.pt"}.issubset(names):\n                    complete.add(directory)\n        for member in saved.namelist():\n            if member.endswith("/"):\n                continue\n            run_file = any(member.startswith(directory + "/") for directory in complete)\n            artifact = any(member.startswith((f"curves/{directory.split(\'/\')[1]}_{directory.split(\'/\')[2]}.",\n                                             f"predictions/{directory.split(\'/\')[1]}_{directory.split(\'/\')[2]}_"))\n                           for directory in complete)\n            if run_file or artifact:\n                target = output / member\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with saved.open(member) as source, target.open("wb") as destination:\n                    shutil.copyfileobj(source, destination)\n                count += 1\n    return count\n\n\ndef restore_completed_tree(source, output):\n    """Restore completed runs when Kaggle extracted the previous evidence zip."""\n    source, output = Path(source), Path(output)\n    count = 0\n    for summary in (source / "runs").glob("*/seed*/summary.json"):\n        directory = summary.parent\n        if not (directory / "config.json").is_file() or not (directory / "best.pt").is_file():\n            continue\n        for path in directory.rglob("*"):\n            if path.is_file():\n                target = output / path.relative_to(source)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                shutil.copy2(path, target)\n                count += 1\n        experiment, seed = directory.parent.name, directory.name\n        for pattern in (f"curves/{experiment}_{seed}.*", f"predictions/{experiment}_{seed}_*"):\n            for path in source.glob(pattern):\n                if path.is_file():\n                    target = output / path.relative_to(source)\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    shutil.copy2(path, target)\n                    count += 1\n    return count\n\n\ndef _config(data, output, **kw):\n    return train.Config(images_dir=str(data / "images"), labels_dir=str(data / "labels"),\n                        out_dir=str(output / "runs"), pred_dir=str(output / "predictions"),\n                        **kw)\n\n\ndef _ckpt_predict(cfg, split, view=None):\n    train_df, val_df, test_df = dataset.load_split(cfg.labels_dir)\n    df = val_df if split == "val" else test_df\n    loader = dataset.make_loader(df, cfg.images_dir,\n                                 dataset.build_transforms(False, cfg.img_size), cfg.batch_size,\n                                 False, num_workers=cfg.num_workers)\n    net = models.build_model(cfg.backbone, pretrained=False, init="scratch", drop_rate=cfg.drop_rate)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    ckpt = torch.load(train.run_dir(cfg) / "best.pt", map_location=device, weights_only=True)\n    net.load_state_dict(ckpt["model"])\n    net.to(device)\n    return inference.predict_logits(net, loader, device, view), net\n\n\ndef _score(y, probs):\n    metric = _eval().compute_metrics(y, probs.argmax(1), probs)\n    return {key: float(metric[key]) for key in ("macro_f1", "top1", "balanced_acc", "ece", "nll")}\n\n\ndef _eda(data, output):\n    frames = dataset.load_split(data / "labels")\n    split = dataset.check_split(*frames, data / "images")\n    _json(output / "evidence" / "split_check.json", split)\n    _json(output / "evidence" / "software.json", {"python": sys.version,\n          "torch": torch.__version__, "torchvision": torchvision.__version__,\n          "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__})\n    manifest = {"images": 17509, "fold": 0, "labels": {}}\n    for name in ("labels.csv", "train_subset0.csv", "val_subset0.csv", "test_subset0.csv"):\n        manifest["labels"][name] = hashlib.sha256((data / "labels" / name).read_bytes()).hexdigest()\n    archive = data / "images.zip"\n    if archive.exists():\n        manifest["images_md5"] = hashlib.md5(archive.read_bytes()).hexdigest()\n    _json(output / "evidence" / "dataset_manifest.json", manifest)\n    counts = pd.DataFrame({part: frame.Label.value_counts().reindex(range(9), fill_value=0)\n                           for part, frame in zip(("train", "val", "test"), frames)})\n    counts.index = dataset.CLASS_NAMES\n    counts.to_csv(output / "evidence" / "class_counts.csv")\n    ax = counts.plot.bar(figsize=(12, 5))\n    ax.set(ylabel="Images", title="DeepWeeds fold 0")\n    plt.tight_layout()\n    plt.savefig(output / "evidence" / "class_counts.png", dpi=160)\n    plt.close()\n    frame = pd.concat(frames, ignore_index=True)\n    fig, axes = plt.subplots(9, 3, figsize=(8, 22))\n    for class_id, row_axes in enumerate(axes):\n        examples = frame[frame.Label == class_id].Filename.head(3)\n        for ax, name in zip(row_axes, examples):\n            with Image.open(data / "images" / name) as im:\n                ax.imshow(im.convert("RGB"))\n            ax.set_title(dataset.CLASS_NAMES[class_id], fontsize=8)\n            ax.axis("off")\n    fig.tight_layout()\n    fig.savefig(output / "evidence" / "sample_images.png", dpi=120)\n    plt.close(fig)\n    return split\n\n\ndef smoke(data, output):\n    output.mkdir(parents=True, exist_ok=True)\n    _eda(data, output)\n    if torch.cuda.is_available():\n        _json(output / "evidence" / "gpu.json", {"count": torch.cuda.device_count(),\n               "names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n               "torch": torch.__version__})\n    cfg = _config(data, output, exp_id="SMOKE", backbone="mobilenetv3_small_050",\n                  init="scratch", epochs=1, batch_size=8, img_size=128, num_workers=2,\n                  max_train_batches=2, max_val_batches=2)\n    result = train.run(cfg)\n    prediction = _eval().read_pred(str(train.pred_path(cfg, "val")))\n    if len(prediction.y_true) != 3501:\n        raise AssertionError("Smoke validation row count mismatch")\n    _json(output / "evidence" / "smoke.json", result)\n    return result\n\n\ndef _inference_candidates(cfgs):\n    ev = _eval()\n    rows, by_seed = [], {}\n    for cfg in cfgs:\n        names, y, base = _ckpt_predict(cfg, "val")[0]\n        flip_names, flip_y, flip = _ckpt_predict(cfg, "val", inference.view_hflip)[0]\n        if names != flip_names or not np.array_equal(y, flip_y):\n            raise AssertionError("TTA changed image order")\n        temp = inference.fit_temperature(base, y)\n        temp_flip = inference.fit_temperature((base + flip) / 2, y)\n        probs = {\n            "I00": inference.apply_temperature(base, 1),\n            "I01": inference.aggregate_views([base, flip], "prob"),\n            "I02": inference.aggregate_views([base, flip], "logit"),\n            "I03": inference.apply_temperature(base, temp),\n            "I04": inference.apply_temperature((base + flip) / 2, temp_flip),\n        }\n        by_seed[cfg.seed] = {"cfg": cfg, "names": names, "y": y, "base": base,\n                              "flip": flip, "temperature": temp,\n                              "temperature_flip": temp_flip, "probs": probs}\n        for method, p in probs.items():\n            rows.append({"exp_id": method, "seed": cfg.seed, **_score(y, p)})\n    frame = pd.DataFrame(rows)\n    means = frame.groupby("exp_id")[["macro_f1", "ece"]].mean()\n    best = means.sort_values(["macro_f1", "ece"], ascending=[False, True]).index[0]\n    return frame, by_seed, best\n\n\ndef _method_probs(method, base, flip, temperature):\n    if method == "I00":\n        return inference.apply_temperature(base, 1)\n    if method == "I01":\n        return inference.aggregate_views([base, flip], "prob")\n    if method == "I02":\n        return inference.aggregate_views([base, flip], "logit")\n    if method == "I03":\n        return inference.apply_temperature(base, temperature)\n    if method == "I04":\n        return inference.apply_temperature((base + flip) / 2, temperature)\n    raise ValueError(method)\n\n\ndef _final_predictions(cfgs, method, temperatures, output):\n    ev = _eval()\n    results = []\n    for cfg in cfgs:\n        (names, y, base), net = _ckpt_predict(cfg, "test")\n        flip = None\n        if method in ("I01", "I02", "I04"):\n            flip_names, flip_y, flip = _ckpt_predict(cfg, "test", inference.view_hflip)[0]\n            if names != flip_names or not np.array_equal(y, flip_y):\n                raise AssertionError("Test TTA image order mismatch")\n        p = _method_probs(method, base, flip, temperatures[cfg.seed])\n        path = train.pred_path(cfg, "test")\n        ev.save_predictions(path, names, y, p)\n        np.savez_compressed(train.run_dir(cfg) / "test_logits.npz", Filename=np.array(names), y_true=y, logits=base)\n        if cfg.exp_id == "F01":\n            uncal_method = {"I03": "I00", "I04": "I02"}.get(method, method)\n            ev.save_predictions(path.with_name(path.stem + "_uncal.csv"), names, y,\n                                _method_probs(uncal_method, base, flip, 1))\n        results.append({"exp_id": cfg.exp_id, "seed": cfg.seed, "method": method, **_score(y, p)})\n    return results\n\n\ndef full(data, output, epochs=10, final_epochs=12):\n    output.mkdir(parents=True, exist_ok=True)\n    split = _eda(data, output)\n    gpu = {"count": torch.cuda.device_count(), "names": [torch.cuda.get_device_name(i)\n           for i in range(torch.cuda.device_count())], "torch": torch.__version__}\n    _json(output / "evidence" / "gpu.json", gpu)\n    if not torch.cuda.is_available():\n        raise RuntimeError("Full lab requires GPU")\n    # Keep the five-family comparison within a Kaggle T4 x2 session. ConvNeXt-Tiny\n    # took ~17 min/epoch in the first Kaggle run and that process was SIGKILLed.\n    backbones = ["resnet50", "resnet18", "efficientnet_b0",\n                 "mobilenetv3_large_100", "vit_tiny_patch16_224"]\n    backbone_rows = []\n    for index, backbone in enumerate(backbones, 1):\n        cfg = _config(data, output, exp_id=f"B{index:02d}", backbone=backbone,\n                      epochs=epochs, batch_size=64,\n                      num_workers=2 if index == 1 else 0)\n        backbone_rows.append(_load_result(cfg))\n    best_backbone = max(backbone_rows, key=lambda r: r["macro_f1_val"])["backbone"]\n    _json(output / "evidence" / "backbone_selection.json",\n          {"criterion": "macro_f1_val", "selected": best_backbone, "candidates": backbone_rows})\n    baseline = _config(data, output, exp_id="T00", backbone=best_backbone,\n                       epochs=epochs, batch_size=64, num_workers=0)\n    training_rows = [_load_result(baseline)]\n    choices = [\n        ("T01", {"init": "frozen"}, "initialization"),\n        ("T02", {"init": "scratch"}, "initialization"),\n        ("T03", {"aug": "color"}, "augmentation"),\n        ("T04", {"aug": "randaug"}, "augmentation"),\n        ("T05", {"loss": "ls", "label_smoothing": 0.1}, "loss"),\n        ("T06", {"loss": "focal", "focal_gamma": 2.0}, "loss"),\n    ]\n    for exp_id, changes, axis in choices:\n        cfg = replace(baseline, exp_id=exp_id, **changes)\n        training_rows.append({**_load_result(cfg), "axis": axis, "change": json.dumps(changes)})\n    # Select each factor on validation only; a combination is then retrained.\n    selected = {}\n    for axis in ("initialization", "augmentation", "loss"):\n        candidates = [r for r in training_rows if r.get("axis") == axis]\n        winner = max(candidates + [training_rows[0]], key=lambda r: r["macro_f1_val"])\n        if winner["exp_id"] != "T00":\n            selected.update(dict(next(c for c in choices if c[0] == winner["exp_id"])[1]))\n    _json(output / "evidence" / "recipe_selection.json", selected)\n    final_cfgs = [replace(baseline, exp_id="F01", seed=seed, epochs=final_epochs, **selected)\n                  for seed in (0, 1, 2)]\n    baseline_cfgs = [replace(baseline, exp_id="F00", seed=seed, epochs=final_epochs)\n                     for seed in (0, 1, 2)]\n    for cfg in baseline_cfgs + final_cfgs:\n        _load_result(cfg)\n    inf_frame, by_seed, method = _inference_candidates(final_cfgs)\n    inf_frame.to_csv(output / "evidence" / "inference_val.csv", index=False)\n    _json(output / "evidence" / "inference_selection.json", {"method": method,\n          "temperatures": {str(k): {"one_view": v["temperature"],\n                                     "two_view": v["temperature_flip"]}\n                           for k, v in by_seed.items()}})\n    for cfg in final_cfgs:\n        info = by_seed[cfg.seed]\n        _eval().save_predictions(train.pred_path(cfg, "val"), info["names"], info["y"],\n                                 info["probs"][method])\n    final_rows = _final_predictions(baseline_cfgs, "I00", {s: 1 for s in (0, 1, 2)}, output)\n    final_rows += _final_predictions(final_cfgs, method,\n                                    {s: by_seed[s]["temperature_flip" if method == "I04" else "temperature"]\n                                     for s in (0, 1, 2)}, output)\n    final_frame = pd.DataFrame(final_rows)\n    final_frame.to_csv(output / "evidence" / "final_scores.csv", index=False)\n    per_class_rows = []\n    for cfg in baseline_cfgs + final_cfgs:\n        pred = _eval().read_pred(str(train.pred_path(cfg, "test")))\n        metric = _eval().compute_metrics(pred.y_true, pred.y_pred, pred.probs)\n        for idx, name in enumerate(dataset.CLASS_NAMES):\n            per_class_rows.append({"exp_id": cfg.exp_id, "seed": cfg.seed, "class": name,\n                                   "support": int(metric["support"][idx]),\n                                   "precision": float(metric["precision"][idx]),\n                                   "recall": float(metric["recall"][idx]),\n                                   "f1": float(metric["f1"][idx])})\n    per_class_frame = pd.DataFrame(per_class_rows)\n    per_class_frame.to_csv(output / "evidence" / "per_class.csv", index=False)\n    _, net = _ckpt_predict(final_cfgs[0], "val")\n    latency_rows = []\n    for batch in (1, 32):\n        for dtype in ("fp32", "amp"):\n            latency_rows.append({"method": "I00", **benchmark.latency_report(net, batch, 224,\n                                  dtype=dtype, device="cuda", iters=50)})\n    two_view = benchmark.tta_latency(net, 2, batch_size=1, img_size=224,\n                                     device="cuda", iters=50)\n    latency_rows.append({"method": "I01/I02/I04", **two_view})\n    latency = pd.DataFrame(latency_rows)\n    latency.to_csv(output / "evidence" / "latency.csv", index=False)\n    one = latency[(latency.method == "I00") & (latency.batch == 1) & (latency.dtype == "fp32")].iloc[0]\n    for metric in ("p50", "p95", "p99"):\n        inf_frame[f"latency_{metric}_ms"] = inf_frame.exp_id.map(\n            {"I00": one[metric], "I01": two_view[metric], "I02": two_view[metric],\n             "I03": one[metric], "I04": two_view[metric]})\n    inf_frame["relative_cost"] = inf_frame.latency_p50_ms / one.p50\n    inf_frame.to_csv(output / "evidence" / "inference_val.csv", index=False)\n    with pd.ExcelWriter(output / "results.xlsx", engine="openpyxl") as writer:\n        pd.DataFrame(backbone_rows).to_excel(writer, sheet_name="Backbones", index=False)\n        pd.DataFrame(training_rows).to_excel(writer, sheet_name="Training", index=False)\n        inf_frame.to_excel(writer, sheet_name="Inference", index=False)\n        final_frame.to_excel(writer, sheet_name="Final", index=False)\n        per_class_frame.to_excel(writer, sheet_name="PerClass", index=False)\n        latency.to_excel(writer, sheet_name="Latency", index=False)\n        summary = pd.concat([pd.DataFrame(backbone_rows), pd.DataFrame(training_rows)])\n        summary.sort_values("macro_f1_val", ascending=False).head(10).to_excel(writer, sheet_name="Summary", index=False)\n        for sheet in writer.sheets.values():\n            sheet.freeze_panes = "A2"\n            sheet.auto_filter.ref = sheet.dimensions\n    final_mean = final_frame[final_frame.exp_id == "F01"].macro_f1.mean()\n    final_std = final_frame[final_frame.exp_id == "F01"].macro_f1.std(ddof=1)\n    base_mean = final_frame[final_frame.exp_id == "F00"].macro_f1.mean()\n    report = (f"# DeepWeeds Lab Day 2\\n\\n"\n              f"## Data and method\\n\\nOriginal DeepWeeds fold 0, 9 classes, {split[\'n\']}. "\n              "Train updates weights; validation chooses checkpoints, recipe and inference; "\n              "test is scored after selection. ImageNet pretrained weights, AdamW, cosine schedule, "\n              "224px input and mixed precision are used unless an experiment config says otherwise. "\n              f"GPU: {\', \'.join(gpu[\'names\'])}; PyTorch {gpu[\'torch\']}.\\n\\n"\n              "## Backbone comparison (validation)\\n\\n"\n              + pd.DataFrame(backbone_rows)[["exp_id", "backbone", "macro_f1_val", "top1_val", "params_m", "gmac", "seconds_per_epoch"]].to_markdown(index=False)\n              + f"\\n\\nSelected: {best_backbone}, highest validation macro-F1.\\n\\n"\n              "## Training recipe (validation)\\n\\n"\n              + pd.DataFrame(training_rows)[["exp_id", "macro_f1_val", "top1_val", "best_epoch"]].to_markdown(index=False)\n              + f"\\n\\nSelected factors by validation: `{json.dumps(selected)}`. "\n              "One factor changes at a time relative to T00; the combination is retrained.\\n\\n"\n              "## Inference selection (validation)\\n\\n"\n              + inf_frame.groupby("exp_id")[["macro_f1", "ece"]].mean().reset_index().to_markdown(index=False)\n              + f"\\n\\nSelected method: {method}. Temperature is fitted on validation only.\\n\\n"\n              "## Final test (three seeds)\\n\\n"\n              + final_frame.to_markdown(index=False)\n              + f"\\n\\nFinal macro-F1: {final_mean:.4f} +/- {final_std:.4f}; "\n              f"baseline: {base_mean:.4f}; delta: {final_mean - base_mean:+.4f}. "\n              "The standard deviation is sample std (ddof=1).\\n\\n"\n              "## Per-class test F1, final configuration\\n\\n"\n              + per_class_frame[per_class_frame.exp_id == "F01"].groupby("class")[["support", "f1"]].mean().reset_index().to_markdown(index=False)\n              + "\\n\\n## Latency\\n\\n"\n              + latency.to_markdown(index=False)\n              + "\\n\\nLatency excludes preprocessing, uses 10 warmup and 50 synchronized iterations. "\n              "See `evidence/`, `runs/`, `curves/`, `predictions/`, and `results.xlsx` for raw evidence. "\n              "Single-seed screening is subject to noise; results may vary across GPU/software versions.\\n")\n    (output / "report.md").write_text(report)\n    ev = _eval()\n    for exp_id in ("F00", "F01"):\n        args = [sys.executable, str(_root() / "eval.py"), "score", "--pred",\n                str(output / "predictions" / f"{exp_id}_seed*_test.csv"),\n                "--test-csv", str(data / "labels" / "test_subset0.csv"),\n                "--out", str(output / "evidence" / f"eval_{exp_id}")]\n        subprocess.run(args, check=True)\n    subprocess.run([sys.executable, str(_root() / "eval.py"), "grade",\n                    "--final", str(output / "predictions" / "F01_seed*_test.csv"),\n                    "--baseline", str(output / "predictions" / "F00_seed*_test.csv"),\n                    "--uncal", str(output / "predictions" / "F01_seed*_test_uncal.csv"),\n                    "--final-val", str(output / "predictions" / "F01_seed*_val.csv"),\n                    "--latency-p95-ms", str(float(latency.iloc[0].p95)),\n                    "--latency-method", "proper",\n                    "--test-csv", str(data / "labels" / "test_subset0.csv"),\n                    "--val-csv", str(data / "labels" / "val_subset0.csv"),\n                    "--out", str(output / "evidence" / "grade")], check=True)\n    _json(output / "evidence" / "completed.json", {"selected_backbone": best_backbone,\n           "selected_recipe": selected, "selected_inference": method, "gpu": gpu})\n    return final_frame\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("phase", choices=("smoke", "full"))\n    parser.add_argument("--data", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--epochs", type=int, default=10)\n    parser.add_argument("--final-epochs", type=int, default=12)\n    args = parser.parse_args()\n    result = smoke(args.data, args.output) if args.phase == "smoke" else full(args.data, args.output,\n                                                args.epochs, args.final_epochs)\n    print(result)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/submissions/2A202602982_HaManhTuan/code/test_lab.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Focused checks for the completed lab code."""\nimport unittest\nimport os\nimport json\nimport tempfile\nimport zipfile\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\n\nimport dataset\nimport inference\nimport losses\nimport model\nimport run_lab\nimport train\n\n\nclass LabTests(unittest.TestCase):\n    def test_original_split_and_image_files(self):\n        data = Path(os.environ.get("DEEPWEEDS_DATA", "/workspace/data"))\n        if not data.exists():\n            self.skipTest("Local dataset not available")\n        frames = dataset.load_split(data / "labels")\n        result = dataset.check_split(*frames, data / "images")\n        self.assertEqual(result["n"], {"train": 10501, "val": 3501, "test": 3507})\n        self.assertFalse(any(result["overlap"].values()))\n\n    def test_focal_zero_equals_ce(self):\n        z = torch.randn(12, 9)\n        y = torch.arange(12) % 9\n        self.assertLess(abs(losses.FocalLoss(gamma=0)(z, y).item() - F.cross_entropy(z, y).item()), 1e-6)\n\n    def test_mix_batch_and_probability_aggregation(self):\n        x, y = torch.randn(8, 3, 16, 16), torch.arange(8)\n        for mode in ("mixup", "cutmix"):\n            mixed, (_, _, lam) = losses.mix_batch(x, y, mode=mode)\n            self.assertEqual(mixed.shape, x.shape)\n            self.assertTrue(0 <= lam <= 1)\n        z = np.random.default_rng(0).normal(size=(8, 9))\n        for space in ("prob", "logit"):\n            p = inference.aggregate_views([z, z], space)\n            np.testing.assert_allclose(p.sum(1), 1, atol=1e-7)\n        self.assertTrue(inference.fit_temperature(z, y.numpy()) > 0)\n\n    def test_model_groups_cover_trainable_parameters(self):\n        net = model.build_model("mobilenetv3_small_050", pretrained=False, init="frozen")\n        groups = model.param_groups(net, 1e-4, 1e-3, 0.05)\n        self.assertEqual(sum(p.numel() for g in groups for p in g["params"]),\n                         sum(p.numel() for p in net.parameters() if p.requires_grad))\n        self.assertTrue(any(not p.requires_grad for p in net.parameters()))\n\n    def test_config_override(self):\n        values = train.parse_overrides(["seed=2", "amp=false", "ema_decay=0.99", "mix=none"])\n        self.assertEqual(values, {"seed": 2, "amp": False, "ema_decay": 0.99, "mix": None})\n\n    def test_conv_bn_fusion_preserves_eval_output(self):\n        net = torch.nn.Sequential(torch.nn.Conv2d(3, 4, 3, padding=1),\n                                  torch.nn.BatchNorm2d(4)).eval()\n        x = torch.randn(2, 3, 8, 8)\n        fused = inference.fuse_conv_bn(net)\n        self.assertLess((net(x) - fused(x)).abs().max().item(), 1e-5)\n\n    def test_resume_rejects_different_epoch_budget(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            cfg = train.Config(out_dir=tmp, epochs=10)\n            folder = train.run_dir(cfg)\n            folder.mkdir(parents=True)\n            (folder / "config.json").write_text(json.dumps(asdict(cfg)))\n            (folder / "summary.json").write_text("{}")\n            with self.assertRaisesRegex(ValueError, "different config"):\n                run_lab._load_result(train.Config(out_dir=tmp, epochs=12))\n\n    def test_scheduler_does_not_advance_on_amp_overflow(self):\n        net = torch.nn.Linear(3, 9)\n        optimizer = torch.optim.SGD(net.parameters(), lr=0.1)\n        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1)\n        class SkippedStep:\n            def __init__(self):\n                self.scale_value = 2.0\n            def scale(self, loss):\n                return loss\n            def get_scale(self):\n                return self.scale_value\n            def step(self, optimizer):\n                pass\n            def update(self):\n                self.scale_value = 1.0\n        loader = [(torch.randn(2, 3), torch.tensor([0, 1]), ["a", "b"])]\n        train.train_one_epoch(net, loader, torch.nn.CrossEntropyLoss(), optimizer,\n                              scheduler, SkippedStep(), train.Config(), torch.device("cpu"))\n        self.assertEqual(scheduler.last_epoch, 0)\n        self.assertEqual(optimizer.param_groups[0]["lr"], 0.1)\n\n    def test_restore_only_completed_runs(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            archive = Path(tmp) / "evidence.zip"\n            with zipfile.ZipFile(archive, "w") as z:\n                for name in ("runs/B01/seed0/config.json", "runs/B01/seed0/summary.json",\n                             "runs/B01/seed0/best.pt", "runs/B02/seed0/best.pt",\n                             "curves/B01_seed0.png", "predictions/B01_seed0_val.csv"):\n                    z.writestr(name, name)\n            destination = Path(tmp) / "out"\n            self.assertEqual(run_lab.restore_completed_runs(archive, destination), 5)\n            self.assertTrue((destination / "runs/B01/seed0/best.pt").exists())\n            self.assertTrue((destination / "curves/B01_seed0.png").exists())\n            self.assertFalse((destination / "runs/B02/seed0/best.pt").exists())\n\n    def test_restored_run_is_reused_without_training(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            output = Path(tmp) / "lab_output"\n            cfg = train.Config(exp_id="B01", out_dir=str(output / "runs"),\n                               pred_dir=str(output / "predictions"), epochs=10)\n            archive = Path(tmp) / "evidence.zip"\n            with zipfile.ZipFile(archive, "w") as z:\n                z.writestr("runs/B01/seed0/config.json", json.dumps(asdict(cfg)))\n                z.writestr("runs/B01/seed0/summary.json", \'{"macro_f1_val": 0.785}\')\n                z.writestr("runs/B01/seed0/best.pt", "checkpoint")\n            run_lab.restore_completed_runs(archive, output)\n            self.assertEqual(run_lab._load_result(cfg)["macro_f1_val"], 0.785)\n\n    def test_restore_extracted_evidence_tree(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            source, output = Path(tmp) / "source", Path(tmp) / "out"\n            folder = source / "runs/B01/seed0"\n            folder.mkdir(parents=True)\n            for name in ("config.json", "summary.json", "best.pt"):\n                (folder / name).write_text(name)\n            (source / "curves").mkdir()\n            (source / "curves/B01_seed0.png").write_text("curve")\n            self.assertEqual(run_lab.restore_completed_tree(source, output), 4)\n            self.assertTrue((output / "runs/B01/seed0/best.pt").exists())\n            self.assertTrue((output / "curves/B01_seed0.png").exists())\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')
print(p)

In [ ]:
from pathlib import Path
import hashlib, urllib.request, zipfile
data = Path('/kaggle/working/data')
(data / 'labels').mkdir(parents=True, exist_ok=True)
base = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels/'
for name in ('labels.csv', 'train_subset0.csv', 'val_subset0.csv', 'test_subset0.csv'):
    target = data / 'labels' / name
    if not target.exists():
        urllib.request.urlretrieve(base + name, target)
archive = data / 'images.zip'
if not archive.exists():
    urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', archive)
digest = hashlib.md5(archive.read_bytes()).hexdigest()
assert digest == 'b7b30f96d466fba86016aa5a26606e0f', digest
images = data / 'images'
if len(list(images.glob('*.jpg'))) != 17509:
    images.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(images)
assert len(list(images.glob('*.jpg'))) == 17509
print('Verified 17,509 images and original fold-0 labels')

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, '/kaggle/working/submissions/2A202602982_HaManhTuan/code')
from run_lab import restore_completed_runs, restore_completed_tree
import zipfile
input_root = Path('/kaggle/input')
archives = []
for candidate in input_root.rglob('*.zip'):
    try:
        with zipfile.ZipFile(candidate) as archive:
            if 'runs/B01/seed0/summary.json' in archive.namelist():
                archives.append(candidate)
    except zipfile.BadZipFile:
        pass
if archives:
    chosen = sorted(archives)[-1]
    restored = restore_completed_runs(chosen, '/kaggle/working/lab_output')
    print(f'Restored {restored} completed-run files from {chosen}')
else:
    summaries = [p for p in input_root.rglob('summary.json')
                 if p.parent.name == 'seed0' and p.parent.parent.name == 'B01'
                 and p.parent.parent.parent.name == 'runs']
    if summaries:
        chosen = sorted(summaries)[-1].parents[3]
        restored = restore_completed_tree(chosen, '/kaggle/working/lab_output')
        print(f'Restored {restored} completed-run files from extracted input {chosen}')
    else:
        print('No previous evidence found in Kaggle Input; all runs will start normally')

In [ ]:
import subprocess, sys
script = '/kaggle/working/submissions/2A202602982_HaManhTuan/code/run_lab.py'
os.environ['DEEPWEEDS_DATA'] = '/kaggle/working/data'
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s',
                '/kaggle/working/submissions/2A202602982_HaManhTuan/code',
                '-p', 'test_*.py', '-v'], check=True)
subprocess.run([sys.executable, script, 'smoke', '--data', '/kaggle/working/data',
                '--output', '/kaggle/working/lab_smoke'], check=True)

In [ ]:
import shutil
try:
    subprocess.run([sys.executable, script, 'full', '--data', '/kaggle/working/data',
                    '--output', '/kaggle/working/lab_output', '--epochs', '10',
                    '--final-epochs', '12'], check=True)
finally:
    if Path('/kaggle/working/lab_output').exists():
        print('Evidence archive:', shutil.make_archive('/kaggle/working/deepweeds_evidence',
                                                     'zip', root_dir='/kaggle/working/lab_output'))

In [ ]:
import shutil
from pathlib import Path
out = Path('/kaggle/working/lab_output')
assert (out / 'evidence' / 'completed.json').is_file()
print('Evidence archive:', '/kaggle/working/deepweeds_evidence.zip')
print('Report:', out / 'report.md')
print('Results:', out / 'results.xlsx')